# Ingestion Inspection

Explore PDF loading, text cleaning, and chunking on the CA DMV Driver Handbook before implementing `src/ingestion/` modules.

**Goals:**
- Load and inspect raw text from the PDF
- Experiment with cleaning (whitespace, encoding, artifacts)
- Try chunk sizes and overlaps; inspect output
- Decide on parameters for `pdf_loader`, `clean_text`, `chunker`

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Project root (parent of notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

PDF_PATH = ROOT / "data" / "ca-drivers-handbook.pdf"
print(f"PDF: {PDF_PATH}")
print(f"Exists: {PDF_PATH.exists()}")

PDF: /Users/srujayreddy/Projects/ca-dmv-rag-system/data/ca-drivers-handbook.pdf
Exists: True


## 2. Load PDF

In [2]:
from pypdf import PdfReader

reader = PdfReader(str(PDF_PATH))
n_pages = len(reader.pages)
print(f"Pages: {n_pages}")

Pages: 92


In [3]:
# Extract text per page (keep page info for later)
pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    pages.append({"page": i + 1, "text": text})

raw_text = "\n\n".join(p["text"] for p in pages)
print(f"Total characters: {len(raw_text):,}")
print(f"Total words (approx): {len(raw_text.split()):,}")

Total characters: 138,237
Total words (approx): 23,459


In [4]:
# Inspect a sample (e.g. first page)
sample = pages[0]["text"]
print("--- First page (first 1500 chars) ---")
print(repr(sample[:1500]))

--- First page (first 1500 chars) ---
'English\nCALIFORNIA\nDRIVER’S HANDBOOK\nThis handbook is available at  \ndmv.ca.gov\nGavin Newsom, Governor \nState of California\nToks Omishakin, Secretary \nCalifornia State Transportation Agency\nSteve Gordon, Director   \nCalifornia Department of Motor Vehicles'


In [5]:
# Look for common artifacts: extra newlines, weird spaces, headers/footers
lines = sample.split("\n")
print(f"Lines on first page: {len(lines)}")
print("First 15 lines:")
for i, line in enumerate(lines[:15]):
    print(f"  {i+1}: {repr(line[:80])}{'...' if len(line) > 80 else ''}")

Lines on first page: 11
First 15 lines:
  1: 'English'
  2: 'CALIFORNIA'
  3: 'DRIVER’S HANDBOOK'
  4: 'This handbook is available at  '
  5: 'dmv.ca.gov'
  6: 'Gavin Newsom, Governor '
  7: 'State of California'
  8: 'Toks Omishakin, Secretary '
  9: 'California State Transportation Agency'
  10: 'Steve Gordon, Director   '
  11: 'California Department of Motor Vehicles'


## 3. Clean text (experiment)

In [6]:
import re

def clean_text(text: str) -> str:
    """Try: collapse whitespace, normalize newlines. Adjust as you find more artifacts."""
    text = re.sub(r"[ \t]+", " ", text)   # multiple spaces/tabs -> single space
    text = re.sub(r"\n{3,}", "\n\n", text)  # 3+ newlines -> 2
    text = text.strip()
    return text

# Apply to first page
cleaned_sample = clean_text(sample)
print("--- After clean (first 800 chars) ---")
print(cleaned_sample[:800])

--- After clean (first 800 chars) ---
English
CALIFORNIA
DRIVER’S HANDBOOK
This handbook is available at 
dmv.ca.gov
Gavin Newsom, Governor 
State of California
Toks Omishakin, Secretary 
California State Transportation Agency
Steve Gordon, Director 
California Department of Motor Vehicles


In [7]:
# Clean full document; compare length
cleaned_full = clean_text(raw_text)
print(f"Raw length:     {len(raw_text):,}")
print(f"Cleaned length: {len(cleaned_full):,}")

Raw length:     138,237
Cleaned length: 138,142


## 4. Chunking (experiment)

In [8]:
def chunk_text(text: str, chunk_size: int = 512, overlap: int = 64) -> list[dict]:
    """Simple sliding window by character count. Returns list of {text, start, end}."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk_text = text[start:end]
        chunks.append({"text": chunk_text, "start": start, "end": end})
        start += chunk_size - overlap
    return chunks

CHUNK_SIZE = 512
CHUNK_OVERLAP = 64
chunks = chunk_text(cleaned_full, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
print(f"Chunks: {len(chunks)} (chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

Chunks: 309 (chunk_size=512, overlap=64)


In [9]:
# Inspect a few chunks
for i in [0, 1, 5, 10]:
    if i < len(chunks):
        c = chunks[i]
        print(f"--- Chunk {i} (start={c['start']}, end={c['end']}) ---")
        print(c["text"][:300] + "..." if len(c["text"]) > 300 else c["text"])
        print()

--- Chunk 0 (start=0, end=512) ---
English
CALIFORNIA
DRIVER’S HANDBOOK
This handbook is available at 
dmv.ca.gov
Gavin Newsom, Governor 
State of California
Toks Omishakin, Secretary 
California State Transportation Agency
Steve Gordon, Director 
California Department of Motor Vehicles

• Renew driver’s license 
 and vehicle registr...

--- Chunk 1 (start=448, end=960) ---
able

i
Dear fellow Californian, 
Whether traveling by car, transit, bike, scooter, 
skateboard or on foot, we all want to reach our 
destination safely. Tragically, many Californians do 
not.
Since 2010, more than 30,000 people have been 
killed and another 100,000 people seriously 
injured on Cali...

--- Chunk 5 (start=2240, end=2752) ---
 Agency

ii
Copyright
© Copyright, Department of Motor Vehicles 2024
All rights reserved.
This work is protected by U.S. Copyright Law. The Department of Motor 
Vehicles (DMV) owns the copyright to this work. Copyright Law makes it 
illegal to:
1. Make a copy of any part of this

In [10]:
# Try different params (optional)
for size, ov in [(256, 32), (512, 64), (1024, 128)]:
    c = chunk_text(cleaned_full, chunk_size=size, overlap=ov)
    print(f"chunk_size={size}, overlap={ov} -> {len(c)} chunks")

chunk_size=256, overlap=32 -> 617 chunks
chunk_size=512, overlap=64 -> 309 chunks
chunk_size=1024, overlap=128 -> 155 chunks


## 5. Chunks with page metadata (optional)

In [11]:
# If we chunk per-page or track char offsets -> page, we can attach page numbers.
# Example: chunk within each page, then add page to metadata.
chunks_with_page = []
for p in pages:
    page_num = p["page"]
    t = clean_text(p["text"])
    for ch in chunk_text(t, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
        ch["page"] = page_num
        chunks_with_page.append(ch)

print(f"Chunks with page: {len(chunks_with_page)}")
print("Sample:", {k: (v if k != "text" else v[:60] + "...") for k, v in chunks_with_page[2].items()})

Chunks with page: 351
Sample: {'text': 'i\nDear fellow Californian, \nWhether traveling by car, transi...', 'start': 0, 'end': 512, 'page': 3}


## 6. Summary & next steps

In [12]:
# Save chunks to JSONL for use in retrieval (optional, for later)
import json

# out = ROOT / "data" / "processed" / "chunks.jsonl"
# out.parent.mkdir(parents=True, exist_ok=True)
# with open(out, "w") as f:
#     for c in chunks_with_page:
#         f.write(json.dumps(c) + "\n")
# print(f"Wrote {out}")

print("Decisions for src/ingestion:")
print("  - pdf_loader: pypdf PdfReader, extract_text per page")
print("  - clean_text: re for whitespace/newlines (add more rules as needed)")
print(f"  - chunker: chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}; include page in metadata")

Decisions for src/ingestion:
  - pdf_loader: pypdf PdfReader, extract_text per page
  - clean_text: re for whitespace/newlines (add more rules as needed)
  - chunker: chunk_size=512, overlap=64; include page in metadata
